# 01 — FMA Acquisition & Curation (Day 1, Core tier)

Downloads the FMA-small subset (8,000 tracks, 8 balanced genres, 30s clips),
stratified-samples it down to ~1,400 tracks, and lands the curated audio + a
`curated_tracks.csv` manifest in Google Drive so this only has to run once.

**Output** (in `MyDrive/SonicExplorer/fma_curated/`):
- `audio/{track_id}.mp3` — ~1,400 curated clips
- `curated_tracks.csv` — track_id, genre_top, title, artist, relative_path

`02_batch_embed_pipeline.ipynb` reads directly from that Drive folder — this
notebook doesn't need to be re-run unless the curated set itself needs to change.

In [ ]:
# --- Mount Drive: curated output needs to survive across Colab sessions ---
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/SonicExplorer/fma_curated'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/audio', exist_ok=True)
print('Drive output folder:', DRIVE_ROOT)

## 1. Download FMA metadata + fma_small

`fma_small.zip` is ~7.2GB — downloaded to Colab's ephemeral `/content` disk (not
Drive) since we only need it transiently to extract the ~1,400 sampled tracks.
`fma_metadata.zip` is small (~342MB) and has the genre labels we need for the
Core-tier genre-cohesion evaluation baseline.

In [ ]:
import requests
from tqdm.auto import tqdm

FMA_SMALL_URL = 'https://os.unil.cloud.switch.ch/fma/fma_small.zip'
FMA_METADATA_URL = 'https://os.unil.cloud.switch.ch/fma/fma_metadata.zip'

def download(url, dest):
    if os.path.exists(dest):
        print(f'{dest} already present, skipping download')
        return
    resp = requests.get(url, stream=True)
    resp.raise_for_status()
    total = int(resp.headers.get('content-length', 0))
    with open(dest, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True, desc=os.path.basename(dest)) as bar:
        for chunk in resp.iter_content(chunk_size=1 << 20):
            f.write(chunk)
            bar.update(len(chunk))

download(FMA_METADATA_URL, '/content/fma_metadata.zip')
download(FMA_SMALL_URL, '/content/fma_small.zip')

## 2. Parse metadata and pick the curated subset

FMA's `tracks.csv` has a two-row header (top-level category + field name).
`track.genre_top` is the single clean genre label per track — the ground truth
the genre-cohesion evaluation compares retrieval neighbors against.

We filter to `set.subset == 'small'` (all 8,000 fma_small tracks, already
genre-balanced at 1,000/genre) and drop any row with a missing genre_top.

In [ ]:
import zipfile
import pandas as pd

with zipfile.ZipFile('/content/fma_metadata.zip') as zf:
    zf.extract('fma_metadata/tracks.csv', '/content')

tracks = pd.read_csv('/content/fma_metadata/tracks.csv', index_col=0, header=[0, 1])

small = tracks[tracks[('set', 'subset')] == 'small'].copy()
small = small[small[('track', 'genre_top')].notna()]

print(f'{len(small)} fma_small tracks with a genre_top label')
print(small[('track', 'genre_top')].value_counts())

## 3. Stratified curation to ~1,400 tracks

Fixed `random_state=42` for reproducibility. Sampling per-genre (not globally)
keeps the curated set genre-balanced, which matters for the evaluation's random
baseline — an unbalanced curated set would skew what "random" means.

In [ ]:
TRACKS_PER_GENRE = 175  # ~175 x 8 genres ~= 1,400 tracks, within the spec's 1,000-2,000 target

# NOTE: groupby(('track','genre_top')).apply(lambda g: g.sample(...)) silently drops
# the genre_top column from the result (a real pandas groupby/apply gotcha, confirmed
# via a local dry-run against just fma_metadata.zip before this was run against the
# full 7.2GB fma_small.zip) -- grouping by an explicit external Series and concatenating
# the per-genre samples avoids it.
genre_series = small[('track', 'genre_top')]
curated_parts = [
    group.sample(n=min(TRACKS_PER_GENRE, len(group)), random_state=42)
    for _, group in small.groupby(genre_series)
]
curated = pd.concat(curated_parts)

print(f'Curated {len(curated)} tracks')
print(curated[('track', 'genre_top')].value_counts())

## 4. Selectively extract only the curated tracks from fma_small.zip

FMA stores each track at `fma_small/{track_id // 1000:03d}/{track_id:06d}.mp3`.
We extract only the ~1,400 sampled files (not all 8,000) directly to the Drive
audio folder, then delete the big zip from `/content` to free disk.

In [ ]:
def fma_zip_path(track_id: int) -> str:
    return f'fma_small/{track_id // 1000:03d}/{track_id:06d}.mp3'

manifest_rows = []
with zipfile.ZipFile('/content/fma_small.zip') as zf:
    for track_id, row in tqdm(curated.iterrows(), total=len(curated), desc='extracting'):
        zip_path = fma_zip_path(track_id)
        dest_name = f'{track_id}.mp3'
        dest_path = f'{DRIVE_ROOT}/audio/{dest_name}'
        if not os.path.exists(dest_path):
            try:
                with zf.open(zip_path) as src, open(dest_path, 'wb') as dst:
                    dst.write(src.read())
            except KeyError:
                print(f'WARNING: {zip_path} not found in zip, skipping track {track_id}')
                continue
        manifest_rows.append({
            'track_id': track_id,
            'genre_top': row[('track', 'genre_top')],
            'title': row[('track', 'title')],
            'artist': row[('artist', 'name')],
            'relative_path': f'audio/{dest_name}',
        })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(f'{DRIVE_ROOT}/curated_tracks.csv', index=False)
print(f'Wrote {len(manifest)} tracks to {DRIVE_ROOT}/curated_tracks.csv')

In [ ]:
# --- Cleanup: free /content disk now that curated audio is safely in Drive ---
for f in ['/content/fma_small.zip', '/content/fma_metadata.zip']:
    if os.path.exists(f):
        os.remove(f)
print('Cleaned up /content zips. Curated audio + manifest persist in Drive.')

## Done

`MyDrive/SonicExplorer/fma_curated/` now holds the curated ~1,400-track library
and its manifest. Next: `02_batch_embed_pipeline.ipynb` clones the
`sonic_explorer` package, reads this manifest, and runs the sound-facet
embedding pipeline (`SongRepository` + `EmbeddingRepository` + `SoundFacet`)
against it.